In [ ]:
!pip install mlflow scikit-learn matplotlib seaborn 
!mlflow --version

In [ ]:
import mlflow
import mlflow.sklearn

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print('All imports OK')

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

print(f'Dataset shape : {X.shape}')   # (1797, 64)
print(f'Classes       : {np.unique(y)}')
print(f'Image size    : 8x8 pixels flattened to 64 features')

# Visualise a few samples
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray_r')
    ax.set_title(f'Label: {digits.target[i]}')
    ax.axis('off')
plt.suptitle('Sample images from the digits dataset', y=1.02)
plt.tight_layout()
plt.savefig('sample_digits.png', bbox_inches='tight')
plt.show()
print('Saved sample_digits.png')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train samples : {len(X_train)}')
print(f'Test samples  : {len(X_test)}')

In [ ]:
# Name your experiment — MLflow creates it if it doesn't exist
mlflow.set_experiment('digit-classifier')

print('Experiment set: digit-classifier')
print('Runs will be saved to ./mlruns/')

mlflow.autolog() -> recommended as it logs every metric, param of each model run by default
include this command before model.start_run()

In [ ]:
with mlflow.start_run(run_name='logistic-regression'):

    # --- Params: anything that defines this run ---
    params = {'model_type': 'LogisticRegression', 'C': 1.0, 'max_iter': 1000, 'solver': 'lbfgs'}
    mlflow.log_params(params)

    # --- Train ---
    model = LogisticRegression(C=params['C'], max_iter=params['max_iter'], solver=params['solver'])
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    # --- Metrics ---
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')
    mlflow.log_metric('accuracy', acc)
    mlflow.log_metric('f1_score', f1)

    # --- Artifact: confusion matrix plot ---
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title('Logistic Regression — Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig('confusion_matrix_lr.png')
    mlflow.log_artifact('confusion_matrix_lr.png')
    plt.show()

    # --- Log the model itself ---
    mlflow.sklearn.log_model(model, name='digit-model')

    print(f'Run complete — Accuracy: {acc:.4f}  |  F1: {f1:.4f}')
    print(f'Run ID: {mlflow.active_run().info.run_id}')

In [ ]:
with mlflow.start_run(run_name='random-forest'):

    params = {'model_type': 'RandomForest', 'n_estimators': 100, 'max_depth': 10, 'random_state': 42}
    mlflow.log_params(params)

    model_rf = RandomForestClassifier(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        random_state=params['random_state']
    )
    model_rf.fit(X_train_scaled, y_train)
    y_pred_rf = model_rf.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred_rf)
    f1  = f1_score(y_test, y_pred_rf, average='weighted')
    mlflow.log_metric('accuracy', acc)
    mlflow.log_metric('f1_score', f1)

    cm = confusion_matrix(y_test, y_pred_rf)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=ax)
    ax.set_title('Random Forest — Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig('confusion_matrix_rf.png')
    mlflow.log_artifact('confusion_matrix_rf.png')
    plt.show()

    mlflow.sklearn.log_model(model_rf, name='digit-model')

    print(f'Run complete — Accuracy: {acc:.4f}  |  F1: {f1:.4f}')
    print(f'Run ID: {mlflow.active_run().info.run_id}')

In [ ]:
with mlflow.start_run(run_name='logistic-regression-high-C'):

    params = {'model_type': 'LogisticRegression', 'C': 10.0, 'max_iter': 1000, 'solver': 'lbfgs'}
    #logging parameters
    mlflow.log_params(params)

    model2 = LogisticRegression(C=params['C'], max_iter=params['max_iter'], solver=params['solver'])
    model2.fit(X_train_scaled, y_train)
    y_pred2 = model2.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred2)
    f1  = f1_score(y_test, y_pred2, average='weighted')
    
    #logging metrics
    mlflow.log_metric('accuracy', acc)
    mlflow.log_metric('f1_score', f1)

    mlflow.sklearn.log_model(model2, name='digit-model')

    print(f'Run complete — Accuracy: {acc:.4f}  |  F1: {f1:.4f}')
    print(f'Run ID: {mlflow.active_run().info.run_id}')

*You can access runs through search_runs(), returns a dataframe*

In [ ]:
runs_df = mlflow.search_runs(experiment_names=['digit-classifier'])

cols = ['tags.mlflow.runName', 'metrics.accuracy', 'metrics.f1_score',
        'params.model_type', 'params.C', 'params.n_estimators']

print(runs_df[cols].sort_values('metrics.accuracy', ascending=False).to_string(index=False))

In [ ]:
# Find the best run automatically
best_run = runs_df.sort_values('metrics.accuracy', ascending=False).iloc[0]
best_run_id   = best_run['run_id']
best_run_name = best_run['tags.mlflow.runName']
best_accuracy = best_run['metrics.accuracy']

print(f'Best run  : {best_run_name}')
print(f'Run ID    : {best_run_id}')
print(f'Accuracy  : {best_accuracy:.4f}')

# Load it back
model_uri = f'runs:/{best_run_id}/digit-model'
loaded_model = mlflow.sklearn.load_model(model_uri)

# Verify it predicts correctly
sample = X_test_scaled[:5]
preds  = loaded_model.predict(sample)
actual = y_test[:5]
print(f'\nSample predictions : {preds}')
print(f'Actual labels      : {actual}')

In [ ]:
!mlflow server --port 5000

In [ ]:
import mlflow

runs = mlflow.search_runs()

print(runs.columns)
print(runs.head())

accessing 